# Clean — CBC Test Dataset

**Vấn đề từ EDA**: giá trị âm/vô lý ngoài khoảng sinh học hợp lý (HGB âm, MCV âm, NEUTp/HCT/MPV vượt xa giới hạn trên).

**Quyết định**: loại bỏ hẳn các dòng vi phạm (không impute) — vì đây là lỗi nhập liệu, không thể suy luận lại giá trị đúng.

**Output**: `Clean_Data/tabular/cbc_test_dataset_clean.csv`

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path(r'D:\AI_08_V1\Data')
OUT = Path(r'D:\AI_08_V1\Clean_Data\tabular')
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA / 'tabular' / 'anemia_cbc_ml_repo' / 'cbc information.xlsx')
print('Shape gốc:', df.shape)

# Khoảng sinh học hợp lý cho toàn bộ 20 chỉ số (dựa trên chuẩn y khoa, có nới rộng để không loại nhầm ca bệnh nặng)
bio_range = {
    'WBC': (0, 100), 'LYMp': (0, 100), 'MIDp': (0, 100), 'NEUTp': (0, 100),
    'LYMn': (0, 50), 'MIDn': (0, 20), 'NEUTn': (0, 50), 'RBC': (0.5, 8),
    'HGB': (2, 22), 'HCT': (5, 70), 'MCV': (40, 150), 'MCH': (10, 50),
    'MCHC': (20, 40), 'RDWSD': (20, 120), 'RDWCV': (8, 30), 'PLT': (0, 1500),
    'MPV': (4, 25), 'PDW': (5, 30), 'PCT': (0, 2), 'PLCR': (0, 100),
}
mask_valid = pd.Series(True, index=df.index)
print('Số dòng vi phạm theo từng cột:')
for col, (lo, hi) in bio_range.items():
    invalid = (df[col] < lo) | (df[col] > hi)
    print(f'  {col:8s}: {invalid.sum():3d} dòng')
    mask_valid &= ~invalid

df_clean = df[mask_valid].reset_index(drop=True)
print(f'\nTổng: {len(df)} dòng gốc -> {len(df_clean)} dòng sạch ({len(df_clean)/len(df)*100:.1f}% giữ lại)')

Shape gốc: (500, 21)
Số dòng vi phạm theo từng cột:
  WBC     :   0 dòng
  LYMp    :   0 dòng
  MIDp    :   0 dòng
  NEUTp   :   2 dòng
  LYMn    :   0 dòng
  MIDn    :   0 dòng
  NEUTn   :   1 dòng
  RBC     :   5 dòng
  HGB     :   8 dòng
  HCT     :   5 dòng
  MCV     :   6 dòng
  MCH     :   7 dòng
  MCHC    :   6 dòng
  RDWSD   :   8 dòng
  RDWCV   :   9 dòng
  PLT     :   0 dòng
  MPV     :   8 dòng
  PDW     :   1 dòng
  PCT     :   4 dòng
  PLCR    :   0 dòng

Tổng: 500 dòng gốc -> 461 dòng sạch (92.2% giữ lại)


In [2]:
# Xác nhận dữ liệu sạch nằm trong khoảng hợp lý + lưu file
print(df_clean[list(bio_range.keys())].describe().T[['min', 'max']])

out_path = OUT / 'cbc_test_dataset_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'\nĐã lưu: {out_path} ({len(df_clean)} dòng)')

         min     max
WBC     0.80   45.70
LYMp    6.20   91.40
MIDp    0.50   77.00
NEUTp   7.60   86.10
LYMn    0.40   41.80
MIDn    0.10    7.00
NEUTn   1.20   44.00
RBC     1.42    6.67
HGB     3.10   17.50
HCT    10.10   53.30
MCV    44.90  106.20
MCH    11.40   34.20
MCHC   20.20   35.70
RDWSD  23.30   57.60
RDWCV  10.60   20.40
PLT    17.00  508.00
MPV     8.00   13.60
PDW     8.40   22.80
PCT     0.01    0.45
PLCR    0.18   48.50

Đã lưu: D:\AI_08_V1\Clean_Data\tabular\cbc_test_dataset_clean.csv (461 dòng)
